In [ ]:
import os
import numpy as np
import laspy
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import random
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import colors as mcolors

In [ ]:
LAZ_DIRECTORY = ""
OUTPUT_DIRECTORY = ""

COLOR_BY = "classification_mapped" # treeID/classification_mapped

In [ ]:
def save_3d_instance_plots(laz_dir, out_dir, color_by="treeID", target_points=15000, point_size=5):
    os.makedirs(out_dir, exist_ok=True)
    files = list(Path(laz_dir).glob('*.laz')) + list(Path(laz_dir).glob('*.las'))
    
    if not files:
        print("Brak plików LAS/LAZ.")
        return

    plt.ioff()

    fixed_colors = {
        0: mcolors.to_rgba('gray'),
        1: mcolors.to_rgba('saddlebrown'),
        2: mcolors.to_rgba('forestgreen')
    }

    for file_path in tqdm(files, desc="Generowanie podglądów 3D"):
        try:
            filename = file_path.name
            stem = file_path.stem

            las = laspy.read(file_path)
            total_points = len(las)
            
            step = max(1, total_points // target_points)
            indices = np.arange(0, total_points, step)
            
            x = las.x[indices]
            y = las.y[indices]
            z = las.z[indices]
            
            try:
                attribute_values = las[color_by][indices]
                has_attribute = True
            except:
                print(f"Brak atrybutu {color_by} w pliku {filename}, kolorowanie po Z.")
                has_attribute = False
                c_array = z

            if has_attribute:
                unique_ids = np.unique(attribute_values)
                
                colormap = plt.get_cmap('nipy_spectral')
                other_colors = [colormap(i) for i in np.linspace(0, 1, len(unique_ids))]
                random.shuffle(other_colors)
                
                color_map_dict = {}
                others_idx = 0
                
                for uid in unique_ids:
                    if uid in fixed_colors:
                        color_map_dict[uid] = fixed_colors[uid]
                    else:
                        color_map_dict[uid] = other_colors[others_idx]
                        others_idx += 1
                        if others_idx >= len(other_colors):
                            others_idx = 0
                
                c_array = [color_map_dict[val] for val in attribute_values]

            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            
            ax.view_init(elev=30, azim=-45)
            
            ax.scatter(x, y, z, c=c_array, s=point_size, marker='s', depthshade=False, edgecolors='none') 
            ax.set_title(filename, fontsize=10)
            
            ax.xaxis.pane.fill = False
            ax.yaxis.pane.fill = False
            ax.zaxis.pane.fill = False
            ax.grid(False)

            max_range = np.array([x.max()-x.min(), y.max()-y.min(), z.max()-z.min()]).max() / 2.0
            mid_x = (x.max()+x.min()) * 0.5
            mid_y = (y.max()+y.min()) * 0.5
            mid_z = (z.max()+z.min()) * 0.5
            ax.set_xlim(mid_x - max_range, mid_x + max_range)
            ax.set_ylim(mid_y - max_range, mid_y + max_range)
            ax.set_zlim(mid_z - max_range, mid_z + max_range)

            output_path = os.path.join(out_dir, f"{stem}.png")
            plt.savefig(output_path, dpi=100, bbox_inches='tight')
            plt.close(fig)

        except Exception as e:
            print(f"Błąd przy pliku {file_path.name}: {e}")
            import traceback
            traceback.print_exc()

    print(f"\nGotowe. Wyniki w: {out_dir}")  
    

In [ ]:
save_3d_instance_plots(LAZ_DIRECTORY, OUTPUT_DIRECTORY, color_by=COLOR_BY, target_points=10000, point_size=15)